# 007 - Stage 5 full: one inverse decoder per model, end to end

**Goal** (from `.github/copilot-instructions.md`): Stage 5 full = Stage 2 with one change: one inverse decoder per model (8) instead of per family.

**Hypothesis.** The instructions give no separate hypothesis for Stage 5 full; this is the pilot's motivation, quoted from the same file: "(Zhao et al., App. V.1: per-model inverses separate two LlamaGen models that share an autoencoder.)" Applied here: each model's signal, computed with that model's own inverse decoder, is lowest for the images that model generated, so the nearest model identifies the size within a family.

**Decision rule** (copied from `.github/copilot-instructions.md`):

> Decision rule (nearest model; the paper's per-model test, App. T, extended to 8 models): for each model m, s_m = the family's selected signal (RAR: quant, VAR: cal, as in Stage 2) with m's own inverse; z_m = (log s_m - mu_m) / sd_m using the task-train images of model m; prediction = argmin_m z_m; outlier if z_min > the 95th percentile of z on that model's own train images. Fit on task train only; evaluate on val once. NearestFamilyThreshold stays unchanged.

**Criteria and prediction** (copied from `.github/copilot-instructions.md`):

> Pre-registered criteria (fixed before results), val:
> - P1 variant accuracy given correct family >= 40% for a family (Stage 2: 25.3%), and every adjacent-size AUC for that family >= 0.60.
> - S1 family accuracy and outlier recall/precision within 2 points of Stage 2 (96.0, 88.0/86.3).
> - Prediction: VAR near chance (pilot); RAR may improve because RAR sizes use different sampling settings.

**Operational definitions** (fixed with the criteria, before any result cell):
- **Rule.** `tracer.stage5.NearestModelThreshold` with `model_score_cols()`: RAR `<model>_quant`, VAR `<model>_cal`, q = 0.95, mean and population std of the log score per model. Fit on the 800 task-train images only. Val is predicted once (section 6), and nothing is changed afterwards.
- **Stage 2 reference.** `NearestFamilyThreshold` is refit exactly as in 002 (Stage 2 features, score = the loss with the highest train AUC per family, q = 0.95, size logistic regression on all 13 features). Section 1 asserts that it reproduces 002's val metrics (accuracy 31.6%, family 96.0%, outlier recall / precision 88.0 / 86.3%, size given family 25.3%), before any comparison.
- **P1, per family f** (RAR and VAR evaluated separately; P1 holds for f if both parts hold):
  1. *Variant accuracy given correct family*: among val images of family f whose predicted family is f, the fraction with the exact label (`size_accuracy_given_family` of `tracer.metrics.report`, restricted to family f). The threshold is ≥ 0.40.
  2. *Adjacent-size AUC*: for each adjacent pair (a, b) in size order (rarb < rarl < rarxl < rarxxl; var16 < var20 < var24 < var30), the val images whose true label is a or b (50 + 50), score z_a − z_b (higher = more like b), positive class b. This is the pairwise AUC of Stage 4 (004). Each of the 3 AUCs must be ≥ 0.60.
- **S1** (one-sided; all three must hold): family accuracy, outlier recall and outlier precision on val are each ≥ the Stage 2 value reproduced in section 1 minus 0.02.
- Pass/fail uses point estimates. 95% bootstrap intervals (val images resampled, 1,000 resamples, seed 0, as in 004) are reported alongside and do not change pass/fail.

**Inputs** (CPU only; reads cached outputs, loads no model): `scripts/run_stage5_full.sh` outputs in `/workspace/cache/stage5full/` (`gen/*/config.json`, `inv/<model>/`, `features/<model>/`); the pilot's var24 set and inverse (`/workspace/cache/stage5/gen`, `inv/var24_full`); Stage 2's features, `inv/<family>/` and `metrics.json`.

**Outputs:** small CSV / JSON / PNG in `results/stage5full/`; val and test predictions in `/workspace/cache/stage5full/` (`submission.csv` is not changed).

**Sections:** 1. Stage 2 reference · 2. Generated training data · 3. Inverse decoders · 4. Features · 5. Fit (train) · 6. Val (once): metrics, criteria, confusion matrices · 7. Exploratory (does not affect pass/fail) · 8. Summary · Findings

In [ ]:
# Setup. CPU only; reads cached outputs, loads no model.
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

REPO = Path("/workspace/iar-model-tracer")
sys.path.insert(0, str(REPO))
from tracer.common import CACHE, FAMILY_OF, LABELS, build_index  # noqa: E402
from tracer.generate import MODELS  # noqa: E402
from tracer.metrics import confusion, family_confusion, report  # noqa: E402
from tracer.signals import ENC2_MIN  # noqa: E402
from tracer.stage1 import KNOWN_FAMILIES, NearestFamilyThreshold  # noqa: E402
from tracer.stage5 import (KNOWN_MODELS, SIGNAL, NearestModelThreshold, load_model_features,  # noqa: E402
                           model_score_cols)

S2 = CACHE / "stage2"  # Stage 2: features/, inv/<family>/, metrics.json
S5P = CACHE / "stage5"  # pilot: var24's training set (gen/) and inverse (inv/var24_full), reused
S5 = CACHE / "stage5full"  # this stage: gen/, inv/<model>/, features/<model>/ (scripts/run_stage5_full.sh)
RES = REPO / "results" / "stage5full"  # small tables and figures, committed
RES.mkdir(parents=True, exist_ok=True)

Q = 0.95  # outlier quantile, fixed in advance as in Stages 1-2
SEED, N_BOOT = 0, 1000
CKPT = {m: S5 / "inv" / m / "final.pt" for m in KNOWN_MODELS}
CKPT["var24"] = S5P / "inv" / "var24_full" / "final.pt"  # identical recipe; not retrained
# Stage 2 val metrics as printed in 002
STAGE2_002 = {"accuracy": 0.316, "family_accuracy": 0.960, "outlier_recall": 0.880, "outlier_precision": 0.863,
              "size_accuracy_given_family": 0.253}
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)


def boot_ci(stat, n, rng, n_boot=N_BOOT):
    """95% percentile interval of stat(idx) over bootstrap resamples idx of range(n) (as in 004)."""
    vals = [stat(rng.integers(0, n, n)) for _ in range(n_boot)]
    return np.percentile(vals, [2.5, 97.5])

## 1. Stage 2 reference (reproduced before any comparison)

Stage 2's rule is refit exactly as in 002 (sections 4–7 there):
- Features: the Stage 2 tables (one shared inverse per family), with the `_greedy` duplicates dropped, leaving 13 features.
- Score per family: the loss with the highest **train** AUC, own family vs the other family.
- `NearestFamilyThreshold` with q = 0.95 and the size logistic regression on all 13 features, fit on train.

The cell asserts that val reproduces 002 (printed values, and `stage2/metrics.json`). Everything below compares against this refit model.

In [ ]:
def load_family_features(d):
    """Join the per-family feature tables on key (as in 002)."""
    rar, var = pd.read_csv(d / "rar.csv"), pd.read_csv(d / "var.csv")
    return rar.merge(var.drop(columns=["split", "label", "family", "name"]), on="key", validate="1:1")


def signed(d, c):
    return d[c].values if c.endswith("tok_match") else -np.log(d[c].values)  # higher = more likely own family


df2 = load_family_features(S2 / "features")
assert df2.key.tolist() == build_index().key.tolist()
FEATS2 = [c for c in df2.columns if c[:4] in ("rar_", "var_") and c not in ("var_quant_greedy", "var_comb_greedy")]
assert np.allclose(df2.var_quant, df2.var_quant_greedy) and np.allclose(df2.var_comb, df2.var_comb_greedy)
train2, val2, test2 = (df2[df2.split == s].reset_index(drop=True) for s in ["train", "val", "test"])

# Score selection as in 002: among the losses, the highest train AUC, own family vs the other family.
SCORE2 = {fam: max((c for c in FEATS2 if c.startswith(fam + "_") and not c.endswith("tok_match")),
                   key=lambda c: roc_auc_score(train2.family == fam, signed(train2, c)))
          for fam in KNOWN_FAMILIES}
saved2 = json.loads((S2 / "metrics.json").read_text())
assert SCORE2 == saved2["score_cols"] == {"rar": "rar_quant", "var": "var_cal"}, SCORE2

model2 = NearestFamilyThreshold(SCORE2, q=Q, size_cols=FEATS2).fit(train2)
pred2_val, pred2_test = model2.predict(val2), model2.predict(test2)
rep2 = report(val2.label, pred2_val)
for k, v in STAGE2_002.items():
    assert abs(rep2[k] - v) < 5e-4, f"{k}: {rep2[k]:.4f}, 002 printed {v}"
    assert np.isclose(rep2[k], saved2["val"][k]), f"{k}: {rep2[k]} vs stage2/metrics.json {saved2['val'][k]}"
print("Stage 2 reproduced on val:", {k: round(rep2[k], 3) for k in STAGE2_002})
print("score columns:", SCORE2, "| tau:", {k: round(v, 3) for k, v in model2.tau_.items()})

## 2. Generated training data

Each model's set was assembled by `assemble_stage5_gen.py` (`gen/<family>/config.json`): 2,560 Stage 2 images (seed 0, A40) and 7,680 new ones (seed 10, H100), 10,240 rows in all. var24's new images are the pilot's (`stage5/gen_extra`), and its links must equal the pilot's training set. Every new source passed the per-family seed check against `stage2/gen`, `stage3/gen` and `stage5/gen_extra`.

In [ ]:
rows = []
for fam in KNOWN_FAMILIES:
    cfg = json.loads((S5 / "gen" / fam / "config.json").read_text())
    assert cfg["labels"] == MODELS[fam] and cfg["seed_scope"] == "family", cfg["labels"]
    for m in MODELS[fam]:
        r = cfg["models"][m]
        base, extra = r["sources"]
        sc = r["seed_check"]
        assert r["n_rows"] == 10240 and (base["seed"], base["n_rows"]) == (0, 2560), m
        assert (extra["seed"], extra["n_rows"]) == (10, 7680), m
        assert base["dir"] == str((S2 / "gen" / fam / m).resolve()), base["dir"]
        assert not sc["class_seed_collision"] and sc["shard_seed_collisions"] == 0 and sc["scope"] == fam, sc
        rows.append({"family": fam, "model": m, "rows": r["n_rows"], "stage2 rows (seed 0)": base["n_rows"],
                     "new rows": extra["n_rows"], "new seed": extra["seed"],
                     "new from": "/".join(Path(extra["dir"]).parts[-4:-2]), "new GPU": extra["gpu"],
                     "class seed": sc["class_seed"], "shard seeds": "..".join(map(str, sc["shard_seeds"])),
                     "seed-check roots": len(sc["roots"])})
gen_tab = pd.DataFrame(rows).set_index(["family", "model"])
display(gen_tab)
gen_tab.to_csv(RES / "gen_composition.csv")


def links(d):
    return {p.name: os.readlink(p) for p in d.iterdir()}


assert links(S5 / "gen" / "var" / "var24") == links(S5P / "gen" / "var" / "var24"), "var24 differs from the pilot"
print("var24: same shard links as the pilot's training set (stage5/gen/var/var24)")

## 3. Inverse decoders

Each per-model inverse must use its family's Stage 2 settings (epochs, batch size, learning rate, StepLR, seed, 100 held out). It is trained on its own model only, on the first 9,940 rows (9,840 training images). The curves come from each run's `log.csv`, where epoch 0 is the original encoder:
- **Held-out columns.** The inverse's own model, last 100 of its 9,940 rows (H100 images). The shared Stage 2 inverse (dashed) is evaluated on a different holdout: the last 100 of each of the family's 4 models' 2,560 A40 images. The two levels are therefore indicative only.
- **Task-train AUC.** AUC of the family's selected signal on the task train images, own family vs the other family, as logged in Stage 2.

In [ ]:
shared_cfg = {fam: json.loads((S2 / "inv" / fam / "config.json").read_text()) for fam in KNOWN_FAMILIES}
SAME = ["epochs", "batch_size", "lr", "step_size", "gamma", "seed", "holdout_per_model", "no_task_train_eval"]
logs = {m: pd.read_csv(CKPT[m].parent / "log.csv") for m in KNOWN_MODELS}
shared_logs = {fam: pd.read_csv(S2 / "inv" / fam / "log.csv") for fam in KNOWN_FAMILIES}
COLS = ["holdout_l_inv", "holdout_tok_acc", "holdout_quant"]

rows = []
for inv in KNOWN_MODELS + KNOWN_FAMILIES:
    fam = FAMILY_OF.get(inv, inv)
    if inv in KNOWN_MODELS:
        cfg, lg = json.loads((CKPT[inv].parent / "config.json").read_text()), logs[inv]
        data = S5P / "gen" if inv == "var24" else S5 / "gen"
        assert CKPT[inv].is_file(), CKPT[inv]
        assert cfg["models"] == [inv] and cfg["max_per_model"] == 9940 and cfg["data"] == str(data), (inv, cfg)
        diff = {k: (cfg[k], shared_cfg[fam][k]) for k in SAME if cfg[k] != shared_cfg[fam][k]}
        assert not diff, (inv, diff)
        name = inv
    else:
        cfg, lg, name = shared_cfg[fam], shared_logs[fam], f"shared {fam} (Stage 2)"
    assert lg.epoch.iloc[-1] == cfg["epochs"], (name, "unfinished run")
    first, last = lg.iloc[0], lg.iloc[-1]
    rows.append({"family": fam, "inverse": name, "epochs": int(last.epoch),
                 **{f"{c} E": first[c] for c in COLS}, **{c: last[c] for c in COLS},
                 "task-train AUC E": first[f"train_auc_{SIGNAL[fam]}"],
                 "task-train AUC": last[f"train_auc_{SIGNAL[fam]}"]})
inv_tab = pd.DataFrame(rows).set_index(["family", "inverse"])
display(inv_tab)
inv_tab.to_csv(RES / "inverse_final.csv")

fig, axs = plt.subplots(2, 3, figsize=(16, 8))
for row, fam in zip(axs, KNOWN_FAMILIES):
    for ax, (c, title) in zip(row, [("holdout_l_inv", "held-out L_inv"), ("holdout_quant", "held-out QuantLoss"),
                                    (f"train_auc_{SIGNAL[fam]}", f"task-train AUC of {SIGNAL[fam]}")]):
        for m in MODELS[fam]:
            ax.plot(logs[m].epoch, logs[m][c], "o-", ms=3, label=m)
        ax.plot(shared_logs[fam].epoch, shared_logs[fam][c], "k--", label="shared (Stage 2)")
        ax.set(title=f"{fam}: {title}", xlabel="epoch (0 = original E)")
        ax.grid(alpha=0.3)
    row[0].set_yscale("log")
    row[1].set_yscale("log")
    row[0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(RES / "inverse_logs.png", dpi=120)
plt.show()

## 4. Features

`load_model_features` joins the 8 per-model tables. It asserts that each was computed with that model's own checkpoint and `--var-iters 0`, and that the rows match `build_index()`. The other checks are those of 002: no NaNs, positive losses, and the number of images whose `enc2` is at the floor.

In [ ]:
df5 = load_model_features(S5 / "features", CKPT)
assert df5.key.tolist() == df2.key.tolist()
SIG5 = [c for c in df5.columns if c.split("_", 1)[0] in KNOWN_MODELS]
assert df5[SIG5].notna().all().all()
assert (df5[[c for c in SIG5 if not c.endswith("tok_match")]] > 0).all().all()
SCORE5 = model_score_cols()
train5, val5, test5 = (df5[df5.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
assert val5.key.tolist() == val2.key.tolist() and test5.key.tolist() == test2.key.tolist()
print({s: len(d) for s, d in [("train", train5), ("val", val5), ("test", test5)]})
print("score columns:", SCORE5)
print("enc2 at floor:", {m: int((df5[f"{m}_enc2"] <= ENC2_MIN).sum()) for m in KNOWN_MODELS})

## 5. Fit on train

For each model m: mu_m and sd_m of log s_m over its own 100 train images, and tau_m, the 95th percentile of z_m over those images.

In [ ]:
model5 = NearestModelThreshold(SCORE5, q=Q).fit(train5)
fit_tab = pd.DataFrame({"family": {m: FAMILY_OF[m] for m in SCORE5}, "score": SCORE5,
                        "mu": model5.mu_, "sd": model5.sd_, "tau": model5.tau_})
display(fit_tab)
fit_tab.to_csv(RES / "fit_train.csv")

## 6. Val, evaluated once

The train-fit rule from section 5 predicts val once. Stage 2 is the refit model from section 1. Metrics are those of `tracer.metrics.report`. Stage 5 full's train metrics are in-sample and shown for reference only.

In [ ]:
pred5_val, pred5_train = model5.predict(val5), model5.predict(train5)
rep5 = report(val5.label, pred5_val)
side = pd.DataFrame({"Stage 2 val": rep2, "Stage 5 full val": rep5})
side["change"] = side["Stage 5 full val"] - side["Stage 2 val"]
side["Stage 5 full train (in-sample)"] = pd.Series(report(train5.label, pred5_train))
display(side)
side.to_csv(RES / "val_metrics.csv")
pd.DataFrame({"key": val5.key, "label": val5.label, "stage2": pred2_val, "stage5full": pred5_val}).to_csv(
    S5 / "val_predictions.csv", index=False)

### Criteria P1 (per family) and S1

The definitions are those in the first cell. Pass/fail uses the point estimate; the bootstrap intervals are for information. The Stage 2 column gives Stage 2's value of the same quantity where one exists. Its size step is a logistic regression, so it has no z-score AUC.

In [ ]:
y = val5.label.values
z5_val = model5.model_z(val5)


def size_correct(pred, fam):
    """Per val image of family fam that was predicted as family fam: exact label correct (bool array)."""
    t, p = pd.Series(y), pd.Series(np.asarray(pred))
    m = (t.map(FAMILY_OF) == fam) & (p.map(FAMILY_OF) == fam)
    return (t[m] == p[m]).values


def pair_aucs(z, sizes, rng):
    """AUC of z_a - z_b (higher = more like b) on val images labelled a or b, for every pair a < b in size order."""
    rows = []
    for i, a in enumerate(sizes):
        for b in sizes[i + 1:]:
            m = np.isin(y, [a, b])
            yb, s = y[m] == b, (z[f"z_{a}"] - z[f"z_{b}"]).values[m]
            lo, hi = boot_ci(lambda r: roc_auc_score(yb[r], s[r]), int(m.sum()), rng)
            gap = sizes.index(b) - i
            rows.append({"family": FAMILY_OF[a], "pair": f"{a} vs {b}", "n": int(m.sum()),
                         "type": "adjacent" if gap == 1 else "extreme" if gap == len(sizes) - 1 else "",
                         "AUC": roc_auc_score(yb, s), "ci_lo": lo, "ci_hi": hi})
    return rows


def boot_rate(x, rng):
    lo, hi = boot_ci(lambda r: x[r].mean(), len(x), rng)
    return x.mean(), lo, hi


rng = np.random.default_rng(SEED)
pairs = pd.DataFrame([r for fam in KNOWN_FAMILIES for r in pair_aucs(z5_val, MODELS[fam], rng)])
crit = []
for fam in KNOWN_FAMILIES:
    c5 = size_correct(pred5_val, fam)
    v, lo, hi = boot_rate(c5, rng)
    crit.append({"criterion": "P1", "family": fam, "quantity": "variant accuracy given correct family",
                 "n": len(c5), "Stage 2": size_correct(pred2_val, fam).mean(), "Stage 5 full": v,
                 "ci_lo": lo, "ci_hi": hi, "threshold": 0.40, "pass": v >= 0.40})
    for _, p in pairs[(pairs.family == fam) & (pairs.type == "adjacent")].iterrows():
        crit.append({"criterion": "P1", "family": fam, "quantity": f"adjacent AUC {p.pair}", "n": p.n,
                     "Stage 2": np.nan, "Stage 5 full": p.AUC, "ci_lo": p.ci_lo, "ci_hi": p.ci_hi,
                     "threshold": 0.60, "pass": p.AUC >= 0.60})

# S1: one-sided, Stage 5 full >= Stage 2 (reproduced) - 0.02. Bootstrap over all val images.
tf, pf5 = val5.family.values, pred5_val.map(FAMILY_OF).values
S1_STATS = {"family_accuracy": lambda r: (tf[r] == pf5[r]).mean(),
            "outlier_recall": lambda r: (pf5[r][tf[r] == "outlier"] == "outlier").mean(),
            "outlier_precision": lambda r: (tf[r][pf5[r] == "outlier"] == "outlier").mean()}
for k, stat in S1_STATS.items():
    lo, hi = boot_ci(stat, len(tf), rng)
    crit.append({"criterion": "S1", "family": "all", "quantity": k, "n": len(tf), "Stage 2": rep2[k],
                 "Stage 5 full": rep5[k], "ci_lo": lo, "ci_hi": hi, "threshold": rep2[k] - 0.02,
                 "pass": rep5[k] >= rep2[k] - 0.02})
crit = pd.DataFrame(crit)
display(crit.set_index(["criterion", "family", "quantity"]))
crit.to_csv(RES / "criteria.csv", index=False)

VERDICT = {f"P1 {fam}": bool(crit[(crit.criterion == "P1") & (crit.family == fam)]["pass"].all())
           for fam in KNOWN_FAMILIES}
VERDICT["S1"] = bool(crit[crit.criterion == "S1"]["pass"].all())
for k, v in VERDICT.items():
    print(f"{k}: {'PASS' if v else 'FAIL'}")

### Confusion matrices (val)

Rows are true labels and columns predicted labels: family level, then all 9 labels, for Stage 2 and Stage 5 full.

In [ ]:
fam_cm = pd.concat({"Stage 2": family_confusion(val2.label, pred2_val),
                    "Stage 5 full": family_confusion(val5.label, pred5_val)}, axis=1)
display(fam_cm)
fam_cm.to_csv(RES / "family_confusion_val.csv")
cms = {"Stage 2": confusion(val2.label, pred2_val), "Stage 5 full": confusion(val5.label, pred5_val)}
for name, cm in cms.items():
    print(name)
    display(cm)
pd.concat(cms, axis=1).to_csv(RES / "confusion_val.csv")

fig, axs = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, (name, cm) in zip(axs, cms.items()):
    ax.imshow(cm.values, cmap="Blues", vmin=0, vmax=50)
    for (i, j), v in np.ndenumerate(cm.values):
        ax.text(j, i, v, ha="center", va="center", fontsize=8, color="white" if v > 30 else "black")
    ax.set(xticks=range(len(cm.columns)), xticklabels=cm.columns, yticks=range(len(cm.index)),
           yticklabels=cm.index, xlabel="predicted", ylabel="true", title=f"{name}, val")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(RES / "confusion_val.png", dpi=120)
plt.show()

## 7. Exploratory (not pre-registered; does not affect pass/fail)

- **E1.** Mean z_m on val, by true label. If the hypothesis holds, the diagonal within each family is the lowest of its row. The table also gives the share of each model's val images whose own model has the lowest z within the true family.
- **E2.** All 6 pairwise AUCs per family, not only the adjacent ones used by P1.
- **E3.** Predicted label shares on test, Stage 5 full vs Stage 2 (refit), next to val. Test labels are hidden, so this is descriptive only and nothing is tuned on it.

In [ ]:
# E1 (exploratory): mean z_m by true val label, and the share of images whose own model is the nearest within the
# true family.
zl = z5_val.groupby(val5.label.values).mean().reindex([l for l in LABELS if l in set(y)])
zl.columns = [c[2:] for c in zl.columns]
own = {}
for fam in KNOWN_FAMILIES:
    zf = z5_val[[f"z_{m}" for m in MODELS[fam]]]
    nearest = np.array(MODELS[fam])[zf.values.argmin(axis=1)]
    for m in MODELS[fam]:
        own[m] = (nearest[y == m] == m).mean()
zl["own model nearest in family"] = pd.Series(own)
display(zl)
zl.to_csv(RES / "z_by_label_val.csv")

fig, ax = plt.subplots(figsize=(9, 6))
zv = zl[KNOWN_MODELS].values
lim = np.nanpercentile(np.abs(zv), 95)
im = ax.imshow(zv, cmap="RdBu_r", vmin=-lim, vmax=lim)
for (i, j), v in np.ndenumerate(zv):
    ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8)
ax.set(xticks=range(len(KNOWN_MODELS)), xticklabels=KNOWN_MODELS, yticks=range(len(zl.index)),
       yticklabels=zl.index, xlabel="model m (own inverse)", ylabel="true label",
       title="val: mean z_m by true label (exploratory)")
fig.colorbar(im, ax=ax, label="mean z_m")
plt.tight_layout()
fig.savefig(RES / "z_by_label_val.png", dpi=120)
plt.show()

In [ ]:
# E2 (exploratory): every pairwise AUC (score z_a - z_b, positive class b), with the bootstrap intervals from above.
display(pairs.set_index(["family", "pair"]))
pairs.to_csv(RES / "pair_auc_val.csv", index=False)

In [ ]:
# E3 (exploratory): predicted label shares on test (descriptive; nothing is tuned on test).
pred5_test = model5.predict(test5)
shares = pd.DataFrame({"test Stage 5 full": pred5_test.value_counts(normalize=True),
                       "test Stage 2": pred2_test.value_counts(normalize=True),
                       "val Stage 5 full": pred5_val.value_counts(normalize=True),
                       "val true": val5.label.value_counts(normalize=True)}).reindex(LABELS).fillna(0)
shares.loc["(family) rar"] = shares.loc[MODELS["rar"]].sum()
shares.loc["(family) var"] = shares.loc[MODELS["var"]].sum()
display(shares)
shares.to_csv(RES / "label_shares.csv")
print(f"test labels that differ from Stage 2: {(pred5_test.values != pred2_test.values).mean():.1%}")
pd.DataFrame({"image_name": test5.name, "stage2": pred2_test, "stage5full": pred5_test}).to_csv(
    S5 / "test_predictions.csv", index=False)

## 8. Summary

Configuration, fitted parameters, val metrics of both stages and the criteria outcome go to `results/stage5full/metrics.json`.

In [ ]:
summary = {"score_cols": SCORE5, "q": Q, "checkpoints": {m: str(p) for m, p in CKPT.items()},
           "mu": model5.mu_, "sd": model5.sd_, "tau": model5.tau_,
           "stage2_reference": {"score_cols": SCORE2, "tau": model2.tau_, "val": rep2},
           "val": rep5, "train_in_sample": report(train5.label, pred5_train), "criteria": VERDICT,
           "bootstrap": {"n_boot": N_BOOT, "seed": SEED}}
(RES / "metrics.json").write_text(json.dumps(summary, indent=2, default=float))
print("criteria:", VERDICT)
print(f"val accuracy: Stage 2 {rep2['accuracy']:.3f}, Stage 5 full {rep5['accuracy']:.3f}")
print("results:", sorted(p.name for p in RES.iterdir()))

## Findings